In [ ]:
## First of all, make sure to check the availability of NVIDIA GPU and CUDA compiler version in the Colab runtime.

!nvidia-smi
!nvcc --version

Tue May 19 01:48:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:

!nvcc -O3 -arch=sm_70 -o matmul matmul.cu  # sm_70 works on most Colab GPUs; change if needed
!./matmul 1024
!./matmul 2048

cc1plus: fatal error: matmul.cu: No such file or directory
compilation terminated.
/bin/bash: line 1: ./matmul: No such file or directory
/bin/bash: line 1: ./matmul: No such file or directory


In [ ]:
!cd
!ls


matmul	matmul.cu  sample_data


In [ ]:
## CELL 1: Setup

import torch, time, math, random
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
torch.manual_seed(SEED); random.seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

Device: cuda


In [ ]:

print(torch.cuda.device_count())  # Number of GPUs


1


In [ ]:
## ROUGH WORK ##

torch.stack([torch.tensor([1,2,3,4]), torch.tensor([1,2,4,3])])

torch.sum(torch.stack([torch.tensor([1,2,3,4]), torch.tensor([1,2,4,3])]), dim=1)

tensor([10, 10])

In [ ]:
# Check the version
print(f"PyTorch version: {torch.__version__}")

NameError: name 'torch' is not defined

In [ ]:
## CELL 2: Self-contained dataset generator: features, labels, COO edges, and masks
from typing import Tuple

# sbm stands for stochastic block model

## Returns a tuple of torch Tensors
def make_sbm(num_nodes: int = 20000, dim: int = 64,
             p_in: float = 0.25, p_out: float = 0.025,
             train_frac: float = 0.6, val_frac: float = 0.2,
             seed: int = 2) -> Tuple[torch.Tensor, ...]:
    """
    This function returns:
      x: [N,F] float32 features
      y: [N] int64 labels in {0,1}
      edge_index: [2,E] int64 COO (undirected, no self-loops)
      train_mask / val_mask / test_mask: [N] bool
    """
    g = torch.Generator().manual_seed(seed)
    N, F = num_nodes, dim
    x = torch.randn(N, F, generator=g)
    y = torch.zeros(N, dtype=torch.long); y[N//2:] = 1

    n0 = N // 2
    rows, cols = [], []
    for i in range(N):
        candidates = torch.randint(0, N, (50,), generator=g)
        for j in candidates.tolist():
            if j <= i:  # avoid self/dups; we'll symmetrize later
                continue
            same = (i < n0 and j < n0) or (i >= n0 and j >= n0)
            p = p_in if same else p_out
            if torch.rand((), generator=g).item() < p:
                rows.append(i); cols.append(j)

    if not rows:
        raise RuntimeError("Graph too sparse; increase p_in/p_out or N.")

    r = torch.tensor(rows, dtype=torch.long)
    c = torch.tensor(cols, dtype=torch.long)
    edge_index = torch.stack([torch.cat([r, c]), torch.cat([c, r])], dim=0)  # undirected

    idx = torch.randperm(N, generator=g)
    n_train = int(train_frac * N)
    n_val = int(val_frac * N)
    train_idx = idx[:n_train]
    val_idx = idx[n_train:n_train+n_val]
    test_idx = idx[n_train+n_val:]

    train_mask = torch.zeros(N, dtype=torch.bool); train_mask[train_idx] = True
    val_mask   = torch.zeros(N, dtype=torch.bool); val_mask[val_idx]   = True
    test_mask  = torch.zeros(N, dtype=torch.bool); test_mask[test_idx] = True
    return x, y, edge_index, train_mask, val_mask, test_mask


In [ ]:
help(make_sbm)

Help on function make_sbm in module __main__:

make_sbm(num_nodes: int = 20000, dim: int = 64, p_in: float = 0.25, p_out: float = 0.025, train_frac: float = 0.6, val_frac: float = 0.2, seed: int = 2) -> Tuple[torch.Tensor, ...]
    Returns:
      x: [N,F] float32 features
      y: [N] int64 labels in {0,1}
      edge_index: [2,E] int64 COO (undirected, no self-loops)
      train_mask / val_mask / test_mask: [N] bool



In [ ]:
## ROUGH WORK ##

import torch.nn.functional as F

# F.tanh()(10)
x = torch.randn(1)
print(x)
F.tanh(x)

tensor([0.0324])


tensor([0.0324])

In [ ]:
## ROUGH WORK ##

## CHECK THE make_sbm() function
# Call with defaults
x, y, edge_index, train_mask, val_mask, test_mask = make_sbm()

# A TOTAL OF 20,000 GRAPH NODES, EACH HAVING F = 64 FEATURES EACH

# Print some basic info
print("Node feature matrix: (x.shape) is", x.shape)        # [N, F]
print("Labels: (y.shape) is", y.shape)                     # [N]
print("Edge index: (edge_index.shape) is", edge_index.shape)        # [2, E]
print("Train/Val/Test sizes:",
      train_mask.sum().item(),
      val_mask.sum().item(),
      test_mask.sum().item())

print(x[1996])

Node feature matrix: (x.shape) is torch.Size([20000, 64])
Labels: (y.shape) is torch.Size([20000])
Edge index: (edge_index.shape) is torch.Size([2, 137148])
Train/Val/Test sizes: 12000 4000 4000
tensor([ 1.5407, -0.0252, -0.6113,  0.8909,  2.4240, -0.6301,  2.0081, -0.1910,
         0.1542, -0.9729, -0.5910, -0.1171, -0.6599, -0.4305, -0.7900, -0.9022,
        -0.3779, -0.5104,  0.6206,  0.7633, -0.3641,  1.3212,  1.0839, -0.5538,
         1.4395, -0.8127, -1.4826,  0.4993, -0.4371,  2.3884,  0.1182,  0.5847,
        -0.9720, -1.2815, -0.2288,  1.2071,  0.1530, -0.0756, -0.5579,  0.6359,
        -0.1888, -0.3712, -0.2560,  0.2886,  1.9457,  0.3887, -0.2525, -0.0474,
        -1.1799, -1.3484,  0.7563,  0.6430, -1.2573,  0.5745, -0.1445, -1.6397,
         0.1329,  0.9161,  0.6279, -1.2487, -0.8627, -1.3262, -1.3297,  0.2494])


In [ ]:
## ROUGH WORK ## (contd.. for make_sbm() function)

print("type(x) is:", type(x))
print("type(x[0] is:", type(x[0]))
print("type(edge_index) is:", type(edge_index))

x[0]

type(x) is: <class 'torch.Tensor'>
type(x[0] is: <class 'torch.Tensor'>
type(edge_index) is: <class 'torch.Tensor'>


tensor([-1.0408,  0.9166, -1.3042, -1.1097, -1.2188,  1.1676, -1.0574, -0.1188,
        -0.9078,  0.3452, -0.5713, -0.2351,  1.0076, -0.7529, -0.2250, -0.4327,
        -1.5071, -0.4586, -0.8480,  0.5266,  0.0299, -0.0498,  1.0651,  0.8860,
         0.4640, -0.4986,  0.1289,  2.7631,  0.1405,  1.1191,  0.3152,  1.7528,
        -0.7650,  1.8299, -0.2784, -0.2719, -1.2944, -0.0243, -0.2354, -0.7087,
         1.1566,  0.4296, -1.1874, -0.7468, -0.9320, -0.8579, -0.9647, -0.0991,
        -1.0190,  0.3157, -1.6036,  1.8493,  0.0447,  1.5853, -0.5912,  1.1312,
         0.7562, -1.2023, -0.5833, -0.4407, -1.9791,  0.7787, -0.7749, -0.1398])

In [ ]:
## ROUGH WORK ##

size = 10
indexes = torch.randperm(size)
print(indexes)
indexes[:6], indexes[6:8], indexes[8:10]


tensor([2, 6, 1, 8, 4, 5, 0, 9, 3, 7])


(tensor([2, 6, 1, 8, 4, 5]), tensor([0, 9]), tensor([3, 7]))

In [ ]:
## ROUGH WORK ##

random_tensor_1 = torch.rand(size=(3, 4))
random_tensor_2 = torch.rand(size=(3, 4))
random_tensor_3 = random_tensor_1 * random_tensor_2 # PyTorch has support for most math operators in Python (+, *, -, /)

random_tensor_1, random_tensor_2, random_tensor_3

(tensor([[0.8823, 0.9150, 0.3829, 0.9593],
         [0.3904, 0.6009, 0.2566, 0.7936],
         [0.9408, 0.1332, 0.9346, 0.5936]]),
 tensor([[0.8694, 0.5677, 0.7411, 0.4294],
         [0.8854, 0.5739, 0.2666, 0.6274],
         [0.2696, 0.4414, 0.2969, 0.8317]]),
 tensor([[0.7670, 0.5195, 0.2837, 0.4119],
         [0.3457, 0.3449, 0.0684, 0.4980],
         [0.2537, 0.0588, 0.2775, 0.4937]]))

In [ ]:
## CELL 3: GraphSAGE-style baseline (pure PyTorch)

def neighbor_mean_agg(x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
    """ out[v] = mean_{u in N(v)} x[u] """
    src, dst = edge_index
    N, _ = x.shape
    out = torch.zeros_like(x)
    out.index_add_(0, dst, x[src])  # sum messages into destinations
    # The above line is the core bottleneck that needs to be solved.

    deg = torch.bincount(dst, minlength=N).clamp(min=1).unsqueeze(1)
    return out / deg

class GraphSageMini(nn.Module):
    def __init__(self, in_dim: int, hidden: int, out_dim: int, dropout: float = 0.2):
        super().__init__()
        self.lin_self1  = nn.Linear(in_dim, hidden, bias=False)
        self.lin_neigh1 = nn.Linear(in_dim, hidden, bias=False)
        self.lin_self2  = nn.Linear(hidden, out_dim,  bias=False)
        self.lin_neigh2 = nn.Linear(hidden, out_dim,  bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index):
        h_neigh = neighbor_mean_agg(x, edge_index)
        h = self.lin_self1(x) + self.lin_neigh1(h_neigh)
        h = F.relu(h)
        h = self.dropout(h)
        h_neigh2 = neighbor_mean_agg(h, edge_index)
        return self.lin_self2(h) + self.lin_neigh2(h_neigh2)

def accuracy(logits, y, mask):
    pred = logits.argmax(dim=-1)
    return (pred[mask] == y[mask]).float().mean().item()


In [ ]:
## CELL 4: Train/Eval loop (timed) of the Baseline

# Hyperparameters
NUM_NODES = 20000
FEAT_DIM  = 64
HIDDEN    = 128
EPOCHS    = 100
LR        = 1e-3
WD        = 1e-4
DROPOUT   = 0.2

# Generating Data
x, y, edge_index, train_mask, val_mask, test_mask = make_sbm(num_nodes=NUM_NODES, dim=FEAT_DIM)
print(x.shape)

x, y, edge_index = x.to(device), y.to(device), edge_index.to(device)
train_mask, val_mask, test_mask = train_mask.to(device), val_mask.to(device), test_mask.to(device)


# Model + optimizer
baseline = GraphSageMini(FEAT_DIM, HIDDEN, 2, dropout=DROPOUT).to(device)
opt = torch.optim.AdamW(baseline.parameters(), lr=LR, weight_decay=WD)

# Train
t0 = time.time()
for epoch in range(1, EPOCHS+1):
    baseline.train(); opt.zero_grad(set_to_none=True)
    logits = baseline(x, edge_index)
    loss = F.cross_entropy(logits[train_mask], y[train_mask])
    loss.backward(); opt.step()
    if epoch % 5 == 0:
        baseline.eval()
        with torch.no_grad():
            val_acc = accuracy(baseline(x, edge_index), y, val_mask)
        print(f"[Baseline] Epoch {epoch:02d} | loss={loss.item():.4f} | val_acc={val_acc:.4f}")
t1 = time.time()

baseline.eval()
with torch.no_grad():
    test_acc = accuracy(baseline(x, edge_index), y, test_mask)
print(f"[Baseline] Time: {t1 - t0:.2f}s | Test acc: {test_acc:.4f}")


torch.Size([20000, 64])
[Baseline] Epoch 05 | loss=0.7003 | val_acc=0.5073
[Baseline] Epoch 10 | loss=0.6877 | val_acc=0.5133
[Baseline] Epoch 15 | loss=0.6787 | val_acc=0.5285
[Baseline] Epoch 20 | loss=0.6729 | val_acc=0.5337
[Baseline] Epoch 25 | loss=0.6665 | val_acc=0.5370
[Baseline] Epoch 30 | loss=0.6603 | val_acc=0.5468
[Baseline] Epoch 35 | loss=0.6519 | val_acc=0.5567
[Baseline] Epoch 40 | loss=0.6448 | val_acc=0.5638
[Baseline] Epoch 45 | loss=0.6404 | val_acc=0.5693
[Baseline] Epoch 50 | loss=0.6302 | val_acc=0.5803
[Baseline] Epoch 55 | loss=0.6220 | val_acc=0.5858
[Baseline] Epoch 60 | loss=0.6118 | val_acc=0.5940
[Baseline] Epoch 65 | loss=0.6028 | val_acc=0.6080
[Baseline] Epoch 70 | loss=0.5915 | val_acc=0.6168
[Baseline] Epoch 75 | loss=0.5839 | val_acc=0.6273
[Baseline] Epoch 80 | loss=0.5690 | val_acc=0.6365
[Baseline] Epoch 85 | loss=0.5556 | val_acc=0.6500
[Baseline] Epoch 90 | loss=0.5478 | val_acc=0.6620
[Baseline] Epoch 95 | loss=0.5334 | val_acc=0.6688
[Baseli

In [ ]:
## CELL 5 — Write C++/CUDA sources

import os, textwrap, pathlib
pathlib.Path("ext/csrc").mkdir(parents=True, exist_ok=True)

open("ext/csrc/gnn_agg.cpp","w").write(textwrap.dedent(r'''
#include <torch/extension.h>
#include <vector>

void agg_forward_cuda(torch::Tensor x, torch::Tensor src, torch::Tensor dst, torch::Tensor out);
void agg_backward_cuda(torch::Tensor grad_out, torch::Tensor src, torch::Tensor dst, torch::Tensor grad_x);

#define CHECK_CUDA(x) TORCH_CHECK(x.is_cuda(), #x " must be CUDA")
#define CHECK_CONTIGUOUS(x) TORCH_CHECK(x.is_contiguous(), #x " must be contiguous")
#define CHECK_INPUT(x) CHECK_CUDA(x); CHECK_CONTIGUOUS(x)

torch::Tensor agg_forward(torch::Tensor x, torch::Tensor src, torch::Tensor dst, int64_t num_nodes) {
  CHECK_INPUT(x); CHECK_INPUT(src); CHECK_INPUT(dst);
  TORCH_CHECK(x.dtype() == torch::kFloat32, "x must be float32");
  TORCH_CHECK(src.dtype() == torch::kLong && dst.dtype() == torch::kLong, "src,dst must be int64");
  auto F = x.size(1);
  auto out = torch::zeros({num_nodes, F}, x.options());
  agg_forward_cuda(x, src, dst, out);
  return out;
}

torch::Tensor agg_backward(torch::Tensor grad_out, torch::Tensor src, torch::Tensor dst, int64_t num_nodes) {
  CHECK_INPUT(grad_out); CHECK_INPUT(src); CHECK_INPUT(dst);
  TORCH_CHECK(grad_out.dtype() == torch::kFloat32, "grad_out must be float32");
  auto F = grad_out.size(1);
  auto grad_x = torch::zeros({num_nodes, F}, grad_out.options());
  agg_backward_cuda(grad_out, src, dst, grad_x);
  return grad_x;
}


// Exposing our C++/CUDA functions defined above to Python as "PyTorch Extensions".
PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
  m.def("agg_forward", &agg_forward, "Neighbor sum forward (CUDA)");
  m.def("agg_backward", &agg_backward, "Neighbor sum backward (CUDA)");
}
'''))

open("ext/csrc/gnn_agg_kernel.cu","w").write(textwrap.dedent(r'''
#include <torch/extension.h>

__global__ void agg_fwd_kernel(const float* __restrict__ x,
                               const long*  __restrict__ src,
                               const long*  __restrict__ dst,
                               float*       __restrict__ out,
                               long E, long F) {
  for (long e = blockIdx.x * blockDim.x + threadIdx.x; e < E; e += blockDim.x * gridDim.x) {
    long s = src[e];
    long d = dst[e];
    const float* xi = x + (long)s * F;
    float*       yo = out + (long)d * F;
    for (long f = 0; f < F; ++f) {
      atomicAdd(yo + f, xi[f]);
    }
  }
}

__global__ void agg_bwd_kernel(const float* __restrict__ grad_out,
                               const long*  __restrict__ src,
                               const long*  __restrict__ dst,
                               float*       __restrict__ grad_x,
                               long E, long F) {
  for (long e = blockIdx.x * blockDim.x + threadIdx.x; e < E; e += blockDim.x * gridDim.x) {
    long s = src[e];
    long d = dst[e];
    const float* go = grad_out + (long)d * F;
    float*       gx = grad_x   + (long)s * F;
    for (long f = 0; f < F; ++f) {
      atomicAdd(gx + f, go[f]);
    }
  }
}

void agg_forward_cuda(torch::Tensor x, torch::Tensor src, torch::Tensor dst, torch::Tensor out) {
  const long E = src.size(0);
  const long F = x.size(1);
  const int threads = 256;
  const int blocks  = (int)min((E + threads - 1) / threads, (long)65535);
  agg_fwd_kernel<<<blocks, threads>>>(x.data_ptr<float>(),
                                      src.data_ptr<long>(),
                                      dst.data_ptr<long>(),
                                      out.data_ptr<float>(),
                                      E, F);
}

void agg_backward_cuda(torch::Tensor grad_out, torch::Tensor src, torch::Tensor dst, torch::Tensor grad_x) {
  const long E = src.size(0);
  const long F = grad_out.size(1);
  const int threads = 256;
  const int blocks  = (int)min((E + threads - 1) / threads, (long)65535);
  agg_bwd_kernel<<<blocks, threads>>>(grad_out.data_ptr<float>(),
                                      src.data_ptr<long>(),
                                      dst.data_ptr<long>(),
                                      grad_x.data_ptr<float>(),
                                      E, F);
}
'''))
print("Wrote ext/csrc/gnn_agg.cpp and gnn_agg_kernel.cu")


Wrote ext/csrc/gnn_agg.cpp and gnn_agg_kernel.cu


In [ ]:
## CELL 6 - Build and bind extension (Autograd Function)
!pip install ninja
from torch.utils.cpp_extension import load

_ext = load(name='gnn_agg_ext',
            sources=['ext/csrc/gnn_agg.cpp','ext/csrc/gnn_agg_kernel.cu'],
            verbose=True) # Removed extra_cuda_cflags


class NeighborSumFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x: torch.Tensor, edge_index: torch.Tensor):
        src, dst = edge_index
        out = _ext.agg_forward(x.contiguous(), src.contiguous(), dst.contiguous(), int(x.size(0)))
        ctx.save_for_backward(src, dst)
        return out
    @staticmethod
    def backward(ctx, grad_out):
        (src, dst,) = ctx.saved_tensors
        grad_x = _ext.agg_backward(grad_out.contiguous(), src.contiguous(), dst.contiguous(), int(grad_out.size(0)))
        return grad_x, None

def neighbor_sum_cuda(x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
    return NeighborSumFunction.apply(x, edge_index)

def neighbor_mean_cuda(x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
    N = x.size(0)
    src, dst = edge_index
    deg = torch.bincount(dst, minlength=N).clamp(min=1).unsqueeze(1)
    return neighbor_sum_cuda(x, edge_index) / deg

print("✅ Built CUDA extension and defined neighbor_mean_cuda()")

KeyboardInterrupt: 

In [ ]:
## CELL 7: Sanity Check Parity vs Python aggregation

# Uses neighbor_mean_agg from Step 1 to verify numerical parity
with torch.no_grad():
    x_chk, _, ei_chk, *_ = make_sbm(num_nodes=5000, dim=16)
    x_chk = x_chk.to(device); ei_chk = ei_chk.to(device)
    ref  = neighbor_mean_agg(x_chk, ei_chk)
    outc = neighbor_mean_cuda(x_chk, ei_chk)
    print("Parity allclose (CUDA vs Python):", torch.allclose(ref, outc, atol=1e-5))
    print("ref: ", ref)
    print("outc: ", outc)

In [ ]:
## CELL 8: Train with CUDA aggregation

class GraphSageCuda(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout=0.2):
        super().__init__()
        self.lin_self1  = nn.Linear(in_dim, hidden, bias=False)
        self.lin_neigh1 = nn.Linear(in_dim, hidden, bias=False)
        self.lin_self2  = nn.Linear(hidden, out_dim,  bias=False)
        self.lin_neigh2 = nn.Linear(hidden, out_dim,  bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index):
        h_neigh = neighbor_mean_cuda(x, edge_index)
        h = self.lin_self1(x) + self.lin_neigh1(h_neigh)
        h = F.relu(h); h = self.dropout(h)
        h_neigh2 = neighbor_mean_cuda(h, edge_index)
        return self.lin_self2(h) + self.lin_neigh2(h_neigh2)

# Data (reuse)
x_c, y_c, ei_c, tr_m, va_m, te_m = make_sbm(num_nodes=NUM_NODES, dim=FEAT_DIM)
x_c, y_c, ei_c = x_c.to(device), y_c.to(device), ei_c.to(device)
tr_m, va_m, te_m = tr_m.to(device), va_m.to(device), te_m.to(device)

# Optional: group edges by dst to reduce atomic contention
order = torch.argsort(ei_c[1])
ei_c = ei_c[:, order]

model_cu = GraphSageCuda(FEAT_DIM, HIDDEN, 2, dropout=DROPOUT).to(device)
opt_cu = torch.optim.AdamW(model_cu.parameters(), lr=LR, weight_decay=WD)

t0 = time.time()
for epoch in range(1, EPOCHS+1):
    model_cu.train(); opt_cu.zero_grad(set_to_none=True)
    logits = model_cu(x_c, ei_c)
    loss = F.cross_entropy(logits[tr_m], y_c[tr_m])
    loss.backward(); opt_cu.step()
    if epoch % 5 == 0:
        model_cu.eval()
        with torch.no_grad():
            val_acc = accuracy(model_cu(x_c, ei_c), y_c, va_m)
        print(f"[CUDA ext] Epoch {epoch:02d} | loss={loss.item():.4f} | val_acc={val_acc:.4f}")
t1 = time.time()

model_cu.eval()
with torch.no_grad():
    test_acc = accuracy(model_cu(x_c, ei_c), y_c, te_m)
print(f"[CUDA ext] Time: {t1 - t0:.2f}s | Test acc: {test_acc:.4f}")


[CUDA ext] Epoch 05 | loss=0.7023 | val_acc=0.5038
[CUDA ext] Epoch 10 | loss=0.6898 | val_acc=0.5053
[CUDA ext] Epoch 15 | loss=0.6798 | val_acc=0.5270
[CUDA ext] Epoch 20 | loss=0.6740 | val_acc=0.5328
[CUDA ext] Epoch 25 | loss=0.6689 | val_acc=0.5383
[CUDA ext] Epoch 30 | loss=0.6606 | val_acc=0.5423
[CUDA ext] Epoch 35 | loss=0.6518 | val_acc=0.5550
[CUDA ext] Epoch 40 | loss=0.6450 | val_acc=0.5648
[CUDA ext] Epoch 45 | loss=0.6351 | val_acc=0.5718
[CUDA ext] Epoch 50 | loss=0.6269 | val_acc=0.5855
[CUDA ext] Epoch 55 | loss=0.6181 | val_acc=0.5963
[CUDA ext] Epoch 60 | loss=0.6038 | val_acc=0.6060
[CUDA ext] Epoch 65 | loss=0.5968 | val_acc=0.6185
[CUDA ext] Epoch 70 | loss=0.5838 | val_acc=0.6278
[CUDA ext] Epoch 75 | loss=0.5739 | val_acc=0.6380
[CUDA ext] Epoch 80 | loss=0.5589 | val_acc=0.6475
[CUDA ext] Epoch 85 | loss=0.5428 | val_acc=0.6563
[CUDA ext] Epoch 90 | loss=0.5318 | val_acc=0.6665
[CUDA ext] Epoch 95 | loss=0.5236 | val_acc=0.6758
[CUDA ext] Epoch 100 | loss=0.5

In [ ]:
## CELL 9: Optimized kernel (feature tiling)

open("ext/csrc/gnn_agg_kernel_opt.cu","w").write(textwrap.dedent(r'''
#include <torch/extension.h>

constexpr int TILE_F = 32;

__global__ void agg_fwd_opt_kernel(const float* __restrict__ x,
                                   const long*  __restrict__ src,
                                   const long*  __restrict__ dst,
                                   float*       __restrict__ out,
                                   long E, long F) {
  for (long e = blockIdx.x * blockDim.x + threadIdx.x; e < E; e += blockDim.x * gridDim.x) {
    long s = src[e];
    long d = dst[e];
    const float* xi = x + (long)s * F;
    float*       yo = out + (long)d * F;
    for (long f0 = 0; f0 < F; f0 += TILE_F) {
      #pragma unroll
      for (int t = 0; t < TILE_F; ++t) {
        long f = f0 + t;
        if (f < F) atomicAdd(yo + f, xi[f]);
      }
    }
  }
}

__global__ void agg_bwd_opt_kernel(const float* __restrict__ grad_out,
                                   const long*  __restrict__ src,
                                   const long*  __restrict__ dst,
                                   float*       __restrict__ grad_x,
                                   long E, long F) {
  for (long e = blockIdx.x * blockDim.x + threadIdx.x; e < E; e += blockDim.x * gridDim.x) {
    long s = src[e];
    long d = dst[e];
    const float* go = grad_out + (long)d * F;
    float*       gx = grad_x   + (long)s * F;
    for (long f0 = 0; f0 < F; f0 += TILE_F) {
      #pragma unroll
      for (int t = 0; t < TILE_F; ++t) {
        long f = f0 + t;
        if (f < F) atomicAdd(gx + f, go[f]);
      }
    }
  }
}

void agg_forward_cuda_opt(torch::Tensor x, torch::Tensor src, torch::Tensor dst, torch::Tensor out) {
  const long E = src.size(0);
  const long F = x.size(1);
  const int threads = 256;
  const int blocks  = (int)min((E + threads - 1) / threads, (long)65535);
  agg_fwd_opt_kernel<<<blocks, threads>>>(x.data_ptr<float>(),
                                          src.data_ptr<long>(),
                                          dst.data_ptr<long>(),
                                          out.data_ptr<float>(),
                                          E, F);
}

void agg_backward_cuda_opt(torch::Tensor grad_out, torch::Tensor src, torch::Tensor dst, torch::Tensor grad_x) {
  const long E = src.size(0);
  const long F = grad_out.size(1);
  const int threads = 256;
  const int blocks  = (int)min((E + threads - 1) / threads, (long)65535);
  agg_bwd_opt_kernel<<<blocks, threads>>>(grad_out.data_ptr<float>(),
                                          src.data_ptr<long>(),
                                          dst.data_ptr<long>(),
                                          grad_x.data_ptr<float>(),
                                          E, F);
}
'''))

open("ext/csrc/gnn_agg_opt.cpp","w").write(textwrap.dedent(r'''
#include <torch/extension.h>

void agg_forward_cuda_opt(torch::Tensor x, torch::Tensor src, torch::Tensor dst, torch::Tensor out);
void agg_backward_cuda_opt(torch::Tensor grad_out, torch::Tensor src, torch::Tensor dst, torch::Tensor grad_x);

#define CHECK_CUDA(x) TORCH_CHECK(x.is_cuda(), #x " must be CUDA")
#define CHECK_CONTIGUOUS(x) TORCH_CHECK(x.is_contiguous(), #x " must be contiguous")
#define CHECK_INPUT(x) CHECK_CUDA(x); CHECK_CONTIGUOUS(x)

torch::Tensor agg_forward_opt(torch::Tensor x, torch::Tensor src, torch::Tensor dst, int64_t num_nodes) {
  CHECK_INPUT(x); CHECK_INPUT(src); CHECK_INPUT(dst);
  auto F = x.size(1);
  auto out = torch::zeros({num_nodes, F}, x.options());
  agg_forward_cuda_opt(x, src, dst, out);
  return out;
}

torch::Tensor agg_backward_opt(torch::Tensor grad_out, torch::Tensor src, torch::Tensor dst, int64_t num_nodes) {
  CHECK_INPUT(grad_out); CHECK_INPUT(src); CHECK_INPUT(dst);
  auto F = grad_out.size(1);
  auto grad_x = torch::zeros({num_nodes, F}, grad_out.options());
  agg_backward_cuda_opt(grad_out, src, dst, grad_x);
  return grad_x;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
  m.def("agg_forward",  &agg_forward_opt,  "Neighbor sum forward (CUDA OPT)");
  m.def("agg_backward", &agg_backward_opt, "Neighbor sum backward (CUDA OPT)");
}
'''))

print("Wrote optimized kernels and bindings")


✅ Wrote optimized kernels and bindings


In [ ]:
## CELL 10: Build optimized Wrapper

_ext_opt = load(name='gnn_agg_ext_opt',
                sources=['ext/csrc/gnn_agg_opt.cpp','ext/csrc/gnn_agg_kernel_opt.cu'],
                verbose=True,
                extra_cuda_cflags=['-O3'])

class NeighborSumFunctionOpt(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x: torch.Tensor, edge_index: torch.Tensor):
        src, dst = edge_index
        out = _ext_opt.agg_forward(x.contiguous(), src.contiguous(), dst.contiguous(), int(x.size(0)))
        ctx.save_for_backward(src, dst)
        return out
    @staticmethod
    def backward(ctx, grad_out):
        (src, dst,) = ctx.saved_tensors
        grad_x = _ext_opt.agg_backward(grad_out.contiguous(), src.contiguous(), dst.contiguous(), int(grad_out.size(0)))
        return grad_x, None

def neighbor_sum_cuda_opt(x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
    return NeighborSumFunctionOpt.apply(x, edge_index)

def neighbor_mean_cuda_opt(x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
    N = x.size(0)
    src, dst = edge_index
    # Cast x to float32 before passing to the CUDA kernel
    x_float32 = x.to(torch.float32)
    deg = torch.bincount(dst, minlength=N).clamp(min=1).unsqueeze(1)
    # Ensure degree is also float32 for division
    return neighbor_sum_cuda_opt(x_float32, edge_index) / deg.to(torch.float32)

print("Built OPT CUDA extension and defined neighbor_mean_cuda_opt()")

✅ Built OPT CUDA extension and defined neighbor_mean_cuda_opt()


In [ ]:
## CELL 11

torch.backends.cuda.matmul.allow_tf32 = True
scaler = torch.cuda.amp.GradScaler(enabled=(device=='cuda'))

class GraphSageCudaOpt(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout=0.2):
        super().__init__()
        self.lin_self1  = nn.Linear(in_dim, hidden, bias=False)
        self.lin_neigh1 = nn.Linear(in_dim, hidden, bias=False)
        self.lin_self2  = nn.Linear(hidden, out_dim,  bias=False)
        self.lin_neigh2 = nn.Linear(hidden, out_dim,  bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, edge_index):
        h_neigh = neighbor_mean_cuda_opt(x, edge_index).to(x.dtype) # Cast to input dtype
        h = self.lin_self1(x) + self.lin_neigh1(h_neigh)
        h = F.relu(h); h = self.dropout(h)
        h_neigh2 = neighbor_mean_cuda_opt(h, edge_index).to(h.dtype) # Cast to input dtype
        return self.lin_self2(h) + self.lin_neigh2(h_neigh2)

# Data (reuse)
x_o, y_o, ei_o, tr_o, va_o, te_o = make_sbm(num_nodes=NUM_NODES, dim=FEAT_DIM)
x_o, y_o, ei_o = x_o.to(device), y_o.to(device), ei_o.to(device)
tr_o, va_o, te_o = tr_o.to(device), va_o.to(device), te_o.to(device)
ei_o = ei_o[:, torch.argsort(ei_o[1])]  # edge reorder

model_opt = GraphSageCudaOpt(FEAT_DIM, HIDDEN, 2, dropout=DROPOUT).to(device)
opt_opt = torch.optim.AdamW(model_opt.parameters(), lr=LR, weight_decay=WD)

t0 = time.time()
for epoch in range(1, EPOCHS+1):
    model_opt.train(); opt_opt.zero_grad(set_to_none=True)
    with torch.cuda.amp.autocast(enabled=(device=='cuda')):
        logits = model_opt(x_o, ei_o)
        loss = F.cross_entropy(logits[tr_o], y_o[tr_o])
    scaler.scale(loss).backward()
    scaler.step(opt_opt)
    scaler.update()
    if epoch % 5 == 0:
        model_opt.eval()
        with torch.no_grad(), torch.cuda.amp.autocast(enabled=(device=='cuda')):
            val_acc = accuracy(model_opt(x_o, ei_o), y_o, va_o)
        print(f"[OPT+AMP] Epoch {epoch:02d} | loss={loss.item():.4f} | val_acc={val_acc:.4f}")
t1 = time.time()

model_opt.eval()
with torch.no_grad():
    test_acc = accuracy(model_opt(x_o, ei_o), y_o, te_o)
print(f"[OPT+AMP] Time: {t1 - t0:.2f}s | Test acc: {test_acc:.4f}")

/tmp/ipython-input-2480578420.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device=='cuda'))
/tmp/ipython-input-2480578420.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device=='cuda')):
/tmp/ipython-input-2480578420.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=(device=='cuda')):


[OPT+AMP] Epoch 05 | loss=0.7054 | val_acc=0.4955
[OPT+AMP] Epoch 10 | loss=0.7005 | val_acc=0.4992
[OPT+AMP] Epoch 15 | loss=0.6982 | val_acc=0.4995
[OPT+AMP] Epoch 20 | loss=0.6960 | val_acc=0.4986
[OPT+AMP] Epoch 25 | loss=0.6949 | val_acc=0.4970
[OPT+AMP] Epoch 30 | loss=0.6933 | val_acc=0.4988
[OPT+AMP] Epoch 35 | loss=0.6923 | val_acc=0.4968
[OPT+AMP] Epoch 40 | loss=0.6914 | val_acc=0.4955
[OPT+AMP] Epoch 45 | loss=0.6906 | val_acc=0.4956
[OPT+AMP] Epoch 50 | loss=0.6900 | val_acc=0.4955
[OPT+AMP] Epoch 55 | loss=0.6895 | val_acc=0.4960
[OPT+AMP] Epoch 60 | loss=0.6887 | val_acc=0.4965
[OPT+AMP] Epoch 65 | loss=0.6879 | val_acc=0.4966
[OPT+AMP] Epoch 70 | loss=0.6872 | val_acc=0.4962
[OPT+AMP] Time: 1.28s | Test acc: 0.4934


In [ ]:
## CELL 12

def bench_once(model, x, ei, y, tr, iters=5):
    optm = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    torch.cuda.synchronize() if device=='cuda' else None
    t0 = time.time()
    for _ in range(iters):
        model.train(); optm.zero_grad(set_to_none=True)
        out = model(x, ei)
        loss = F.cross_entropy(out[tr], y[tr])
        loss.backward(); optm.step()
    torch.cuda.synchronize() if device=='cuda' else None
    return time.time() - t0

# Fresh data (bigger graph → clearer speed diffs)
xB, yB, eiB, trB, vaB, teB = make_sbm(num_nodes=100_000, dim=64, p_in=0.0008, p_out=0.00008)
xB, yB, eiB = xB.to(device), yB.to(device), eiB.to(device)
trB = trB.to(device)
eiB_sorted = eiB[:, torch.argsort(eiB[1])]

b_base = GraphSageMini(64, 128, 2).to(device)
b_cu   = GraphSageCuda(64, 128, 2).to(device)
b_opt  = GraphSageCudaOpt(64, 128, 2).to(device)

tb  = bench_once(b_base, xB, eiB, yB, trB)
tcu = bench_once(b_cu,   xB, eiB_sorted, yB, trB)
topt= bench_once(b_opt,  xB, eiB_sorted, yB, trB)

print(f"Baseline   : {tb:.2f}s for 5 iters")
print(f"CUDA ext   : {tcu:.2f}s for 5 iters")
print(f"OPT + AMP  : {topt:.2f}s for 5 iters  (AMP is enabled in Cell 11 training loop)")

Baseline   : 0.06s for 5 iters
CUDA ext   : 0.06s for 5 iters
OPT + AMP  : 0.05s for 5 iters  (AMP is enabled in Cell 11 training loop)


In [ ]:
## CELL 12

def bench_once(model, x, ei, y, tr, iters=5):
    optm = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    torch.cuda.synchronize() if device=='cuda' else None
    t0 = time.time()
    for _ in range(iters):
        model.train(); optm.zero_grad(set_to_none=True)
        out = model(x, ei)
        loss = F.cross_entropy(out[tr], y[tr])
        loss.backward(); optm.step()
    torch.cuda.synchronize() if device=='cuda' else None
    return time.time() - t0

# Fresh data (bigger graph → clearer speed diffs)
xB, yB, eiB, trB, vaB, teB = make_sbm(num_nodes=100_000, dim=64, p_in=0.0008, p_out=0.00008)
xB, yB, eiB = xB.to(device), yB.to(device), eiB.to(device)
trB = trB.to(device)
eiB_sorted = eiB[:, torch.argsort(eiB[1])]

b_base = GraphSageMini(64, 128, 2).to(device)
b_cu   = GraphSageCuda(64, 128, 2).to(device)
b_opt  = GraphSageCudaOpt(64, 128, 2).to(device)

tb  = bench_once(b_base, xB, eiB, yB, trB)
tcu = bench_once(b_cu,   xB, eiB_sorted, yB, trB)
topt= bench_once(b_opt,  xB, eiB_sorted, yB, trB)

print(f"Baseline   : {tb:.2f}s for 5 iters")
print(f"CUDA ext   : {tcu:.2f}s for 5 iters")
print(f"OPT + AMP  : {topt:.2f}s for 5 iters  (AMP is enabled in Cell 11 training loop)")


Baseline   : 0.06s for 5 iters
CUDA ext   : 0.06s for 5 iters
OPT + AMP  : 0.06s for 5 iters  (AMP is enabled in Cell 11 training loop)
